In [ ]:
import numpy as np

import pygambit as gbt


def create_repeated_game_efg(
    A: np.ndarray,
    B: np.ndarray,
    T: int,
    title: str | None = None,
) -> gbt.Game:
    """Create a T-period repeated game from a simultaneous m*n stage game.

    At every round, each Player 1 decision node opens a proper subgame, so
    the game contains ((m*n)^T - 1) / (m*n - 1) subgame roots in total.

    Parameters
    ----------
    A : np.ndarray
        Payoff matrix for Player 1 (m*n).
    B : np.ndarray
        Payoff matrix for Player 2 (m*n).
    T : int
        Number of repetitions (>= 1).
    title : str, optional

    Returns
    -------
    Game
        The extensive-form repeated game with:
        - (m*n)^T terminal nodes
        - (1 + m) * ((m*n)^T - 1) / (m*n - 1) non-terminal nodes
        - 2 * ((m*n)^T - 1) / (m*n - 1) information sets
        - ((m*n)^T - 1) / (m*n - 1) subgame roots

    Notes
    -----
    Terminal outcomes are shared across leaves whose accumulated payoffs are equal:
    leaves with identical `(sum_A, sum_B)` along their path point to a single
    `Outcome` object.  Sharing happens by exact equality of cache keys,
    so for float-valued matrices, paths whose sums differ only by floating-point
    round-off will produce distinct outcome objects with semantically identical values.
    If exact outcome sharing matters, coerce `A` and `B` to an exact type before calling:

        from fractions import Fraction
        A = np.array([[Fraction(x) for x in row] for row in A], dtype=object)
        B = np.array([[Fraction(x) for x in row] for row in B], dtype=object)

    For integer-valued matrices this caveat does not apply.
    """
    assert A.shape == B.shape
    assert A.shape[0] >= 1 and A.shape[1] >= 1
    assert T >= 1

    m, n = A.shape
    if title is None:
        title = f"Repeated {m}x{n} game ({T} periods)"
    g = gbt.Game.new_tree(players=["1", "2"], title=title)
    actions1 = [str(i) for i in range(m)]
    actions2 = [str(i) for i in range(n)]

    payoffs_to_outcomes = {}
    frontier = [(g.root, 0, 0)]
    for t in range(1, T + 1):
        next_frontier = []
        for node, acc_a, acc_b in frontier:
            g.append_move(node, "1", actions1)
            p1_children = list(node.children)
            g.append_move(p1_children, "2", actions2)
            for i, p1_child in enumerate(p1_children):
                for j, child in enumerate(p1_child.children):
                    new_acc_a = acc_a + A[i, j]
                    new_acc_b = acc_b + B[i, j]
                    if t == T:
                        key = (new_acc_a, new_acc_b)
                        if key not in payoffs_to_outcomes:
                            payoffs_to_outcomes[key] = g.add_outcome(
                                [new_acc_a, new_acc_b]
                            )
                        g.set_outcome(child, payoffs_to_outcomes[key])
                    else:
                        next_frontier.append((child, new_acc_a, new_acc_b))
        frontier = next_frontier
    return g


def create_binary_two_player_efg(
    level: int,
    title: str | None = None,
) -> gbt.Game:
    """Create a 2-player binary extensive-form game of given depth.

    Players 1 and 2 alternate moves starting with player 1 at the root.
    At every depth >= 1, any two non-terminal siblings share a single
    information set, so the moving player cannot tell which of their
    opponent's two prior actions they're responding to.
    All leaves have payoff [0, 0] as payoffs do not matter for subgame detection.

    Parameters
    ----------
    level : int
        Depth of the binary tree (>= 1).
    title : str, optional

    Returns
    -------
    Game
        The extensive-form game with:
        - 2 ** level terminal nodes
        - 2 ** level - 1 non-terminal nodes
        - Player 1 has the move at every even depth, Player 2 at every odd depth
        - The root forms a singleton infoset
        - At every depth >= 1, each parent's pair of children shares one infoset
        - A single shared outcome [0, 0] at every leaf
    """
    assert level >= 1

    if title is None:
        title = f"Binary 2-player game (L={level})"
    g = gbt.Game.new_tree(players=["1", "2"], title=title)
    actions = ["L", "R"]

    # Depth 0: root is a singleton infoset for player 1.
    g.append_move(g.root, "1", actions)
    frontier = [g.root]

    # Depths 1..level-1: each parent's pair of children gets one infoset
    # via a single `append_move` call passing the pair.
    for depth in range(1, level):
        player_label = str((depth % 2) + 1)
        new_frontier = []
        for parent in frontier:
            children = list(parent.children)
            g.append_move(children, player_label, actions)
            new_frontier.extend(children)
        frontier = new_frontier

    # All leaves share one zero outcome.
    zero_outcome = g.add_outcome([0, 0])
    for parent in frontier:
        for leaf in parent.children:
            g.set_outcome(leaf, zero_outcome)

    return g


def create_centipede_efg(
    N: int,
    title: str | None = None,
) -> gbt.Game:
    """Create a Centipede game of N rounds with zero payoffs everywhere.

    Players 1 and 2 alternate moves starting with player 1 at the root.
    At each round, the moving player chooses between "Take" (terminating
    the game with a payoff) and "Push" (passing the decision to the other player).
    After N rounds, the "Push" branch terminates the game with a payoff.
    All leaves have payoff [0, 0] as payoffs do not matter for subgame detection.

    Centipede is a perfect-information game, so all infosets are singletons.

    Parameters
    ----------
    N : int
        Number of decision rounds (>= 1).
    title : str, optional

    Returns
    -------
    Game
        The extensive-form Centipede game with:
        - N + 1 terminal nodes (N "Take" leaves plus one "Push" leaf at the end)
        - N decision nodes
        - N singleton information sets
        - A single shared outcome [0, 0] at every leaf
    """
    assert N >= 1

    if title is None:
        title = f"Centipede game ({N} rounds)"
    g = gbt.Game.new_tree(players=["1", "2"], title=title)
    actions = ["Take", "Push"]
    zero_outcome = g.add_outcome([0, 0])

    current = g.root
    for t in range(N):
        player = str((t % 2) + 1)
        g.append_move(current, player, actions)
        take_child, push_child = current.children
        g.set_outcome(take_child, zero_outcome)
        if t == N - 1:
            g.set_outcome(push_child, zero_outcome)
        else:
            current = push_child

    return g

In [ ]:
import statistics
import time

_Z22 = np.zeros((2, 2), dtype=int)
_Z23 = np.zeros((2, 3), dtype=int)
_Z33 = np.zeros((3, 3), dtype=int)

SUBGAME_BENCHMARKS = [
    # Centipede
    ("centipede-N300",     lambda: create_centipede_efg(N=300)),
    ("centipede-N3000",    lambda: create_centipede_efg(N=3000)),
    ("centipede-N30000",   lambda: create_centipede_efg(N=30000)),
    ("centipede-N150000",  lambda: create_centipede_efg(N=150000)),
    # Binary
    ("binary-2p-depth-5",  lambda: create_binary_two_player_efg(level=5)),
    ("binary-2p-depth-10", lambda: create_binary_two_player_efg(level=10)),
    ("binary-2p-depth-15", lambda: create_binary_two_player_efg(level=15)),
    ("binary-2p-depth-18", lambda: create_binary_two_player_efg(level=18)),
    # Repeated
    ("repeated-2x2-T3",    lambda: create_repeated_game_efg(_Z22, _Z22, T=3)),
    ("repeated-2x3-T4",    lambda: create_repeated_game_efg(_Z23, _Z23, T=4)),
    ("repeated-3x3-T5",    lambda: create_repeated_game_efg(_Z33, _Z33, T=5)),
    ("repeated-2x3-T7",    lambda: create_repeated_game_efg(_Z23, _Z23, T=7)),
]


def benchmark_subgame_detection(benchmarks, n_runs=1000):
    for name, factory in benchmarks:
        game = factory()
        n_nodes = len(game.nodes)
        n_infosets = len(game.infosets)

        times = []
        for _ in range(n_runs):
            game = factory()
            start = time.perf_counter()
            _ = game.root.is_subgame_root
            elapsed = time.perf_counter() - start
            times.append(elapsed)

        times.sort()
        median_ms = statistics.median(times) * 1000
        q1 = times[len(times) // 4] * 1000
        q3 = times[3 * len(times) // 4] * 1000
        semi_iqr = (q3 - q1) / 2

        print(f"{name}  (nodes={n_nodes}, infosets={n_infosets})")
        print(
            f"  Median: {median_ms:.3f} ± {semi_iqr:.3f} ms ({median_ms/n_nodes*1000:.2f} µs/node)"
        )
        print(f"  IQR: [{q1:.3f}, {q3:.3f}] ms")
        print()


benchmark_subgame_detection(SUBGAME_BENCHMARKS)